In [5]:

import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/Jan/Reanalysis/Our_Analysis")
META = ROOT / "02_session_metadata"
RESULTS = ROOT / "04_results"
RESULTS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(META / "recovered_24_eligible_regions.csv")

genotypes = ["WT", "Fmr1", "Cntnap2", "Shank3B"]

# Anatomically defined regions for the primary analysis.
region_groups = {
    "DG": ["DG-mo", "DG-sg", "DG-po"],
    "CA1": ["CA1"],
    "CA3": ["CA3"],
    "VISp": ["VISp2/3", "VISp4", "VISp5", "VISp6a", "VISp6b"],
    "MOs": ["MOs2/3", "MOs5", "MOs6a"],
    "CP": ["CP"],
    "SUB": ["SUB"],
    "VL": ["VL"],
    "MRN": ["MRN"],
    "APN": ["APN"],
}

# Parent atlas labels and white-matter structures.
excluded_regions = ["HPF", "STR", "MB", "ccb", "cing", "dhc"]

records = []

for group, regions in region_groups.items():
    subset = df[df["region"].isin(regions)]

    row = {
        "region_group": group,
        "subregions": ", ".join(regions),
        "n_subregions": len(subset)
    }

    for genotype in genotypes:
        animals = subset[f"{genotype}_animals"]
        units = subset[f"{genotype}_units"]
        events = subset[f"{genotype}_complete_event_mice"]

        # Without mouse IDs, the exact union across
        # subregions cannot be calculated.
        row[f"{genotype}_mice_lower_bound"] = int(animals.max())
        row[f"{genotype}_units"] = int(units.sum())
        row[f"{genotype}_event_mice_lower_bound"] = int(events.max())

    row["min_mice_lower_bound"] = min(
        row[f"{g}_mice_lower_bound"] for g in genotypes
    )

    row["min_event_mice_lower_bound"] = min(
        row[f"{g}_event_mice_lower_bound"] for g in genotypes
    )

    records.append(row)

grouped = pd.DataFrame(records).sort_values(
    "min_mice_lower_bound", ascending=False
)

grouped["eligible_all_genotypes"] = (
    grouped["min_mice_lower_bound"] >= 5
)

# Save the new analysis matrix permanently.
grouped.to_csv(
    RESULTS / "anatomical_group_coverage.csv",
    index=False
)

df[df["region"].isin(excluded_regions)].to_csv(
    RESULTS / "excluded_atlas_labels.csv",
    index=False
)

display_cols = (
    ["region_group"]
    + [f"{g}_mice_lower_bound" for g in genotypes]
    + [f"{g}_units" for g in genotypes]
    + ["min_event_mice_lower_bound"]
)

print("ANATOMICAL GROUP COVERAGE")
display(grouped[display_cols])

print("\nELIGIBLE GROUPS:")
print(grouped.loc[
    grouped["eligible_all_genotypes"], "region_group"
].tolist())

print("\nSaved to:", RESULTS / "anatomical_group_coverage.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ANATOMICAL GROUP COVERAGE


,region_group,WT_mice_lower_bound,Fmr1_mice_lower_bound,Cntnap2_mice_lower_bound,Shank3B_mice_lower_bound,WT_units,Fmr1_units,Cntnap2_units,Shank3B_units,min_event_mice_lower_bound
0,DG,15,13,13,13,1654,2592,865,1552,12
1,CA1,13,11,11,14,1134,1495,1117,1879,11
2,CA3,11,10,10,9,568,324,771,666,8
4,MOs,12,12,8,7,1096,1519,986,979,6
5,CP,9,10,7,6,822,890,807,1070,5
8,MRN,9,11,6,7,2033,1136,1280,796,6
3,VISp,13,11,5,9,1285,1174,594,1039,5
6,SUB,8,10,6,5,808,1293,681,797,5
7,VL,7,8,5,7,131,154,172,567,5
9,APN,7,10,5,7,622,1000,314,586,5



ELIGIBLE GROUPS:
['DG', 'CA1', 'CA3', 'MOs', 'CP', 'MRN', 'VISp', 'SUB', 'VL', 'APN']

Saved to: /content/drive/MyDrive/Jan/Reanalysis/Our_Analysis/04_results/anatomical_group_coverage.csv


In [6]:

from pathlib import Path

folders = [
    Path("/content/drive/MyDrive/Jan/Reanalysis/Our_Analysis"),
    Path("/content/autism_anatomy_inventory"),
]

target_files = {
    "insertion_region_details.csv",
    "region_genotype_details.csv",
    "region_genotype_matrix.csv",
}

for folder in folders:
    print(f"\nChecking: {folder}")
    if not folder.exists():
        print("Not found")
        continue

    found = [
        p for p in folder.rglob("*.csv")
        if p.name in target_files
    ]

    for p in found:
        print("FOUND:", p)

    if not found:
        print("No detailed inventory files found")



Checking: /content/drive/MyDrive/Jan/Reanalysis/Our_Analysis
No detailed inventory files found

Checking: /content/autism_anatomy_inventory
Not found


In [7]:

import pandas as pd
from pathlib import Path
from io import StringIO

ROOT = Path("/content/drive/MyDrive/Jan/Reanalysis/Our_Analysis")
META = ROOT / "02_session_metadata"
DOCS = ROOT / "00_documentation"

META.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

# Allen CCFv3 anatomical region dictionary.
region_text = """allen_acronym|full_name|macro_system|analysis_region|analysis_status
DG-mo|Dentate gyrus, molecular layer|Hippocampal formation|DG|include
DG-sg|Dentate gyrus, granule cell layer|Hippocampal formation|DG|include
DG-po|Dentate gyrus, polymorph layer|Hippocampal formation|DG|include
CA1|Field CA1|Hippocampal formation|CA1|include
CA3|Field CA3|Hippocampal formation|CA3|include
SUB|Subiculum|Hippocampal formation|SUB|include
VISp2/3|Primary visual area, layer 2/3|Cerebral cortex|VISp|include
VISp4|Primary visual area, layer 4|Cerebral cortex|VISp|include
VISp5|Primary visual area, layer 5|Cerebral cortex|VISp|include
VISp6a|Primary visual area, layer 6a|Cerebral cortex|VISp|include
VISp6b|Primary visual area, layer 6b|Cerebral cortex|VISp|include
MOs2/3|Secondary motor area, layer 2/3|Cerebral cortex|MOs|include
MOs5|Secondary motor area, layer 5|Cerebral cortex|MOs|include
MOs6a|Secondary motor area, layer 6a|Cerebral cortex|MOs|include
CP|Caudoputamen (dorsal striatum)|Striatum|CP|include
MRN|Midbrain reticular nucleus|Midbrain|MRN|include
APN|Anterior pretectal nucleus|Midbrain|APN|include
HPF|Hippocampal formation, unspecified|Unresolved parent label|UNRESOLVED_HPF|exclude_primary
STR|Striatum, unspecified|Unresolved parent label|UNRESOLVED_STR|exclude_primary
MB|Midbrain, unspecified|Unresolved parent label|UNRESOLVED_MB|exclude_primary
VL|Lateral ventricle|Ventricular system|NON_NEURAL|exclude
ccb|Corpus callosum, body|Fiber tracts|NON_NEURAL|exclude
cing|Cingulum bundle|Fiber tracts|NON_NEURAL|exclude
dhc|Dorsal hippocampal commissure|Fiber tracts|NON_NEURAL|exclude
"""

atlas = pd.read_csv(StringIO(region_text), sep="|")

# Save the anatomical dictionary.
atlas.to_csv(META / "allen_region_dictionary.csv", index=False)

# Read the previously recovered region counts.
coverage = pd.read_csv(
    META / "recovered_24_eligible_regions.csv"
)

# Attach region labels without changing original counts.
annotated = coverage.merge(
    atlas,
    left_on="region",
    right_on="allen_acronym",
    how="left",
    validate="one_to_one"
)

annotated.to_csv(
    META / "annotated_region_coverage.csv",
    index=False
)

# Save the analysis plan.
plan = """# Electrophysiological reanalysis plan

Dataset: angelaki_mouseASD
Genotypes: WT, Fmr1, Cntnap2, Shank3B

Primary neuronal regions:
DG, CA1, CA3, SUB, VISp, MOs, CP, MRN, APN.

1. Recover mouse/session/probe/cluster identifiers and Allen labels.
2. Merge anatomical sublayers at the individual-cluster level.
3. Select QC-pass neuronal units and report unique mouse coverage.
4. Calculate firing rate, ISI CV and CV2.
5. Quantify bursting and pause characteristics.
6. Analyze stimulus-, movement- and feedback-aligned activity.
7. Compare each ASD genotype with WT.
8. Assess shared and genotype-specific abnormalities.

Statistics:
Mouse is the biological replicate.
Retain neuron, probe and session identifiers.
Report effect sizes, confidence intervals and adjusted p-values.
Apply multiple-comparison correction to confirmatory tests.

Important:
VL denotes lateral ventricle, not ventrolateral thalamus.
White matter and unspecified parent regions are excluded
from primary neuronal comparisons.

All long computations must save checkpoints to Google Drive.
"""

(DOCS / "ANALYSIS_PLAN.md").write_text(plan, encoding="utf-8")

print("Saved anatomical dictionary:", len(atlas), "labels")
print("Primary neuronal regions:",
      atlas.loc[atlas.analysis_status == "include",
                "analysis_region"].nunique())
print("Saved annotated coverage and analysis plan.")

display(atlas)


Saved anatomical dictionary: 24 labels
Primary neuronal regions: 9
Saved annotated coverage and analysis plan.


,allen_acronym,full_name,macro_system,analysis_region,analysis_status
0,DG-mo,"Dentate gyrus, molecular layer",Hippocampal formation,DG,include
1,DG-sg,"Dentate gyrus, granule cell layer",Hippocampal formation,DG,include
2,DG-po,"Dentate gyrus, polymorph layer",Hippocampal formation,DG,include
3,CA1,Field CA1,Hippocampal formation,CA1,include
4,CA3,Field CA3,Hippocampal formation,CA3,include
5,SUB,Subiculum,Hippocampal formation,SUB,include
6,VISp2/3,"Primary visual area, layer 2/3",Cerebral cortex,VISp,include
7,VISp4,"Primary visual area, layer 4",Cerebral cortex,VISp,include
8,VISp5,"Primary visual area, layer 5",Cerebral cortex,VISp,include
9,VISp6a,"Primary visual area, layer 6a",Cerebral cortex,VISp,include


In [9]:

%pip -q install one-api ibllib

from one.api import ONE

ROOT = Path("/content/drive/MyDrive/Jan/Reanalysis/Our_Analysis")
META = ROOT / "02_session_metadata"
META.mkdir(parents=True, exist_ok=True)

one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    username="intbrainlab",
    password="international",
    silent=True
)

# Retrieve probe insertions belonging to the ASD dataset.
insertions = one.alyx.rest(
    "insertions",
    "list",
    django="session__projects__name,angelaki_mouseASD"
)

records = []

for insertion in insertions:
    session = insertion.get("session")
    if not isinstance(session, dict):
        continue

    subject = session.get("subject", "")
    if isinstance(subject, dict):
        subject = subject.get("nickname", "")

    subject = str(subject)
    prefix = subject.split("-")[0].upper()

    genotype = {
        "NYU": "WT",
        "FMR": "Fmr1",
        "CSP": "Cntnap2",
        "SH": "Shank3B"
    }.get(prefix)

    if genotype is None:
        continue

    records.append({
        "pid": insertion["id"],
        "eid": session["id"],
        "probe": insertion.get("name"),
        "subject": subject,
        "genotype": genotype
    })

inventory = pd.DataFrame(
    records,
    columns=["pid", "eid", "probe", "subject", "genotype"]
).drop_duplicates(subset=["pid"])

inventory.to_csv(META / "mouse_probe_inventory.csv", index=False)

print("Total probe insertions:", len(inventory))
print("Unique mice:", inventory["subject"].nunique())
display(inventory.groupby("genotype").agg(
    mice=("subject", "nunique"),
    probes=("pid", "nunique")
))
print("Saved:", META / "mouse_probe_inventory.csv")


/usr/local/lib/python3.13/dist-packages/one/webclient.py:1053: RuntimeWarning: The shared public account will be retired soon. Register your own account at https://openalyx.internationalbrainlab.org/signup
  emit(message)


Total probe insertions: 0
Unique mice: 0


,mice,probes
genotype,,


Saved: /content/drive/MyDrive/Jan/Reanalysis/Our_Analysis/02_session_metadata/mouse_probe_inventory.csv
